# Linear models
Linear models are one of the most simple ML methods intended for regression and classification tasks, in which target value is expected to be a linear combination of features. For instance, the single variable case represents a first degree polynomial:

$$
    y(x) = mx + b
$$

where $b$ is the intercept and $m$ is the slope of the straight. In this post I want to share you what linear models exists, how are used, and how model training is performed for each one.

## Linear regression
Linear regression is the most basic approach for a regression task, here we want to find linear model that fits a continuos dependant variable $y$. In this case we are considering that our model has more than 1 independent variable, so our input, independent coefficient, slope and dependant variable will be vectors:

$$
    y^{*}(\vec{x}) = \vec{w} \cdot \vec{x} + b
$$

it's the same definition but instead of scalars we have vectors, notice that we renamed the slope coefficient $m$ to $\vec{w}$ vector that represents the weights for each features telling us how much they contribute to the model output.

### How training is performed?
The training stage consists on find the vectors $\vec{w}$ and $\vec{b}$ that fits the target variable $\vec{y}$, the default approach is least squares method. Ordinary least squares is an analytical approach to find the vectors that minimizes the quadratic error function for the model outcomes. The scalar definition is given as follows:
$$
    J(\vec{y^{*}}) = \sum_{i=1}^{n} (\vec{y}_i - \vec{y^*}_i)^2
$$

where:
- $n$ is the number of samples.
- $y_i$ is the output for sample $i$.
- $\vec{y^{*}}$ is the vector containing the outputs of the model.

Square error function measures how far are the model outomes to the true sample value, the difference is squared to avoid term cancellation among errors of different inputs, resulting in $J(\vec{y^{*}}) = 0$ and then interpreted to "hey, we have the perfect fit, we don't need to find a better solution".

#### Ordinary least squares
This is a numerical approach to find an analytical solution for finding a vector $\vec{w}$ that minimizes $J(\vec{y^{*}})$ error function, let's start re-writting the loss function in terms of $w_i$ coefficients:

$$
  J(\vec{y^{*}}) = \sum_{i=1}^{n} (\vec{y}_i - b - \sum_{j=1}^{p} \vec{w_i} \vec{x_{ij}})^2
$$

The procedure is the following:

1. Differrentiate $J(\vec{y^{*}})$ error function by $\vec{w}$ components.

$$
    \frac{\partial J}{\partial w_j} = (-2\sum_{i=1}^{n} x_{ij})(\sum_{i=1}^n \vec{y}_i - b - \sum_{k=1}^{p} \vec{x}_{ik} \vec{w}_{k})
$$

2. We need to find the minimum so we equalize the  the partial derivate to 0.

$$
    0 = (-2\sum_{i=1}^{n} x_{ij})(\sum_{i=1}^n \vec{y}_i - b - \sum_{k=1}^{p} \vec{x}_{ik} \vec{w}_{k})
$$

3. Building up the product and reordering.

$$
    b \sum_{i=1}^{n} \vec{x}_{ij} + \sum_{k=1}^{p} \vec{w}_{k} \sum_{i=1}^{n} \vec{x}_{ij} \vec{x}_{ik} = \sum_{i=1}^{n} \vec{x}_{ij} \vec{y}_{i}
$$

We got a linear system of $p+1$ equations, one for the intercept and other ones for each variable weight. This approach give us the exact solution if exists, however the computations are expensive because at high level we are performing matrix operations like calculating inverse matrixes.

#### Stochastic descent gradient (SGD)
Analytical solution is fine but takes many computational resources, so a more efficient approach consists on using SGD method defining an initial solution and iterating to reduce the error. The squared error function is differentiable, let's update the vectorial definition of the model

$$
    y_{i}^{*}(\vec{x}) = \vec{w} \cdot \vec{x}
$$

having:
- $\vec{w}_0 = b$ to calculate the intercept coefficient.
- $\vec{x}_0 = 1$, represents the intercept existence.
- $||\vec{w}|| = ||\vec{x}|| = p + 1$

as we seen in [neural networks guide](https://www.kaggle.com/code/criser2013/neural-networks) we use the negative gradient of the loss function to find the direction where it comes minimal. The procedure is similar, wee need to differentiate the loss function by the model output, choose an initial solution (often a zero or random vector), select a learning rate and start iterating. The learning rule is defined as follows:

$$
    w^{(i+1)} = w^{(i)} - \eta \frac{\partial J}{\partial w}
$$

and we can add regularization coefficients or early stopping to avoid overfiting or performance downgrade. This method has a complexity of $O(kn \bar{p})$ where $k$ is the number of training epochs, $n$ the number of samples and $\bar{p}$ the average number of non-zero attributes per sample, so it has a linear cost.

### Assumptions
- Variables $x$ and $y$ have a linear relationship.
- The residuals are independent of each other, their covariance is 0.
- The residuals has a constant variance.
- The residuals follows a Gaussian distribution.

### Limitations
- Sensitive to outliners
- Assumes that the chosen model appropriately represents the relationship between the variables.

## Logistic regression

In [ ]:
from pandas import read_csv
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, f1_score

data = read_csv("/kaggle/input/datasets/iabhishekofficial/mobile-price-classification/train.csv")
COL_MAPPER = {"fc": "front_camera_pixels", "four_g": "has_4g", "int_memory": "storage_size", 
              "mobile_wt": "weight", "m_dep": "depth_cm", "n_cores": "cpu_cores", "pc": "rear_camera_pixels",
              "px_width": "screen_width_pixels", "px_height": "screen_height_pixels", "sc_w": "screen_width_cm",
              "sc_h": "screen_height_cm", "three_g": "has_3g", "touch_screen": "has_touch_screen", "wifi": "has_wifi",
              "blue": "has_bluetooth", "clock_speed": "cpu_clock_speed"}

renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["battery_power","cpu_clock_speed", "depth_cm", "front_camera_pixels", "storage_size", "weight", "cpu_cores",
       "rear_camera_pixels", "screen_height_pixels","screen_width_pixels","ram","screen_height_cm","screen_width_cm","talk_time"]
BINS = ["has_4g", "has_3g", "has_bluetooth", "dual_sim", "has_wifi", "has_touch_screen"]
FEATURES = NUMS + BINS
TAG = "price_range"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

TRANSFORMER = ColumnTransformer([
    ("standarized", StandardScaler(), NUMS)
], remainder="passthrough")

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
CLASSIFICATION_MODEL = MLPClassifier(hidden_layer_sizes=(20, 15, 10, 5), solver="lbfgs", activation="relu", random_state=123)
CLASSIFICATION_MODEL.fit(X_train, y_train)
y_pred = CLASSIFICATION_MODEL.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1Score = f1_score(y_test, y_pred, average="weighted")
print(f"Accuracy: {(accuracy*100):2f} %")
print(f"Weighted F1-Score: {(f1Score*100):2f} %")

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = read_csv("/kaggle/input/datasets/jsonali2003/mobile-price-prediction-dataset/Mobile Price Prediction Datatset.csv")

data = data.drop(columns=["Unnamed: 0"])
COL_MAPPER = {"Brand me": "brand", "Ratings": "user_ratings", "RAM": "ram", 
              "ROM": "storage_size", "Mobile_Size": "display_size_inches", "Primary_Cam": "main_camera_px", "Selfi_Cam": "selfie_camera_px",
              "Battery_Power": "battery", "Price": "price"}
renamed_data = data.rename(columns=COL_MAPPER)

NUMS = ["user_ratings","ram", "storage_size", "display_size_inches", "main_camera_px", "selfie_camera_px", "battery"]
CATS = ["brand"]
FEATURES = NUMS + CATS
TAG = "price"

x_train, x_test, y_train, y_test = train_test_split(renamed_data[FEATURES], renamed_data[TAG], train_size=0.8, random_state=123)

CAT_TRANSFORMER = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

TRANSFORMER = ColumnTransformer([
    ("num", KNNImputer(), NUMS),
    ("cat", CAT_TRANSFORMER, CATS)
])

X_train = TRANSFORMER.fit_transform(x_train)
X_test = TRANSFORMER.transform(x_test)

# Model definition
REGRESSION_MODEL = MLPRegressor(hidden_layer_sizes=(8,8,8,4,2), activation="identity", solver="lbfgs", random_state=123)
REGRESSION_MODEL.fit(X_train, y_train)
y_pred = REGRESSION_MODEL.predict(X_test)

squared_error = mean_squared_error(y_test, y_pred)
absolute_error = mean_absolute_error(y_test, y_pred)
print(f"Mean squared error: {squared_error:2f}")
print(f"Mean absolute error: {absolute_error:2f}")

## Most common hyperparameters

## Advantages
- a

## Disadvantages
- a